# Huấn luyện Mô hình Seq2Seq LSTM + Bahdanau Attention trên Kaggle

**Lưu ý thiết lập quan trọng trước khi chạy:**
1. Ở cột bên phải (**Settings**):
   - **Accelerator**: Chọn **GPU T4 x2** hoặc **GPU P100** (miễn phí 30 giờ/tuần).
   - **Internet**: Chuyển sang **ON** (bắt buộc để tải thư viện và dữ liệu).

In [ ]:
# Kiểm tra card đồ họa GPU
!nvidia-smi

## 1. Lấy mã nguồn từ GitHub về Kaggle

- Nếu repository **Public**: Chạy lệnh clone thông thường.
- Nếu repository **Private**: Điền GitHub Personal Access Token (PAT) vào URL bên dưới.

In [ ]:
# Cách 1: Nếu repo là Public
# !git clone -b Hiếu https://github.com/vha06/dich-may-Anh-Viet.git

# Cách 2: Nếu repo là Private (điền token của bạn thay cho <GITHUB_TOKEN>)
# GITHUB_TOKEN = "ghp_xxxxxx"
# !git clone -b Hiếu https://{GITHUB_TOKEN}@github.com/vha06/dich-may-Anh-Viet.git

# Hoặc clone thẳng nhánh Hiếu:
!git clone -b Hiếu https://github.com/vha06/dich-may-Anh-Viet.git

## 2. Di chuyển vào thư mục dự án và cài đặt thư viện

In [ ]:
%cd /kaggle/working/dich-may-Anh-Viet
!pip install -q -r requirements.txt

## 3. Tải và xử lý dữ liệu tự động (Tuần 1 Pipeline)

In [ ]:
# Tải 6 file dữ liệu IWSLT15 En-Vi về data/raw/
!python scripts/download_data.py

# Tiền xử lý, chuẩn hóa Unicode NFC, lọc độ dài về data/processed/
!python src/data/preprocess.py

# Huấn luyện BPE Tokenizer (8000 vocab) về data/tokenizer/
!python src/data/tokenizer.py

## 4. Huấn luyện mô hình Seq2Seq LSTM + Attention

Quá trình train sẽ chạy 20 epoch. Mô hình có điểm BLEU cao nhất trên tập val (tst2012) sẽ tự động được lưu vào `checkpoints/best_seq2seq.pt`.

In [ ]:
!python scripts/train.py --config configs/baseline_lstm.yaml

## 5. Kiểm tra checkpoint và Dịch thử câu thực tế

In [ ]:
import torch
from src.data.tokenizer import BPETokenizer
from src.models.seq2seq_lstm import Encoder, Decoder, Seq2Seq
from src.evaluation.decode import greedy_decode, beam_search_decode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tok = BPETokenizer("data/tokenizer/bpe.model")

enc = Encoder(vocab_size=8000, embed_dim=256, hidden_dim=512, num_layers=2, dropout=0.3, pad_id=tok.pad_id)
dec = Decoder(vocab_size=8000, embed_dim=256, encoder_dim=1024, hidden_dim=512, attention_dim=256, num_layers=2, dropout=0.3, pad_id=tok.pad_id)
model = Seq2Seq(enc, dec, pad_id=tok.pad_id).to(device)

ckpt = torch.load("checkpoints/best_seq2seq.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"✓ Đã nạp checkpoint Epoch {ckpt['epoch']} (Best BLEU = {ckpt['bleu']:.2f})")

test_sentences = [
    "How are you today ?",
    "I am learning machine translation with deep learning .",
    "Vietnam is a beautiful country in Southeast Asia .",
    "Thank you very much for your help ."
]

print("\n=== KẾT QUẢ DỊCH THỰC TẾ ===")
for s in test_sentences:
    src_ids = torch.tensor([tok.encode(s, add_bos=True, add_eos=True)], device=device)
    greedy_res, _ = greedy_decode(model, src_ids, tok, device=device)
    beam_res = beam_search_decode(model, src_ids, tok, beam_width=5, device=device)
    print(f"\n[EN]: {s}")
    print(f"[Greedy]: {greedy_res[0]}")
    print(f"[Beam  ]: {beam_res[0]}")